## Environment and Protocol Setup

In [1]:
# ============================================================
# Cell 1 — Environment setup, imports, and project discovery
# ============================================================

import os
import sys
import time
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# ------------------------------------------------------------
# Project Directory Path Resolution
# ------------------------------------------------------------
# Use an explicit environment variable when available.
# This provides a deterministic way to specify the project code location.
CODE_DIR_ENV = os.environ.get("THESIS_CODE_DIR")

if CODE_DIR_ENV:
    # Resolve the path to an absolute, normalized location.
    CODE_DIR = Path(CODE_DIR_ENV).resolve()

else:
    candidates = []

    # Check the current working directory for the required project modules.
    cwd = Path.cwd()
    if (
        (cwd / "data_pipeline.py").exists()
        and (cwd / "experiment_protocol.py").exists()
    ):
        candidates.append(cwd)

    # If running on Kaggle, search the input directory recursively.
    kaggle_input = Path("/kaggle/input")

    if kaggle_input.exists():
        # Find all possible data pipeline modules.
        pipeline_files = list(
            kaggle_input.rglob("data_pipeline.py")
        )

        for pipeline_file in pipeline_files:
            candidate_dir = pipeline_file.parent

            # A valid project directory must contain both required modules.
            if (candidate_dir / "experiment_protocol.py").exists():
                candidates.append(candidate_dir)

    # Remove duplicate paths while preserving their discovery order.
    candidates = list(
        dict.fromkeys(
            path.resolve()
            for path in candidates
        )
    )

    # Require exactly one valid project directory to avoid ambiguous imports.
    if len(candidates) == 1:
        CODE_DIR = candidates[0]

    elif len(candidates) == 0:
        # Stop execution if the project modules cannot be located.
        raise FileNotFoundError(
            "Could not locate data_pipeline.py and "
            "experiment_protocol.py in the same directory.\n\n"
            "Expected structure:\n"
            "/kaggle/input/<dataset>/.../data_pipeline.py\n"
            "/kaggle/input/<dataset>/.../experiment_protocol.py\n\n"
            "Set THESIS_CODE_DIR explicitly if necessary."
        )

    else:
        # Prevent the notebook from importing modules from an unintended
        # project when multiple valid directories are detected.
        raise RuntimeError(
            "Multiple valid code directories were found.\n\n"
            + "\n".join(
                str(path)
                for path in candidates
            )
            + "\n\n"
            "Set THESIS_CODE_DIR explicitly to the "
            "directory you want to use."
        )


# ------------------------------------------------------------
# System Path Injection
# ------------------------------------------------------------
# Add the resolved project directory to the beginning of sys.path.
# This ensures that the local project modules take precedence during import.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


# ------------------------------------------------------------
# Import Custom Project Modules
# ------------------------------------------------------------
# Load the shared data-loading function used by the experiment.
from data_pipeline import load_prepared_experiment

# Import the centralized experimental protocol.
# These constants define datasets, sample sizes, seeds, models,
# evaluation settings, and other parameters used consistently
# across the baseline experiments.
from experiment_protocol import (
    PROTOCOL_VERSION,
    EXPECTED_DATASETS,
    TRAINING_SIZES,
    SUBSET_SEEDS,
    TEST_SIZE,
    TEST_SEED,
    MODEL_SEED,
    CLASSIFICATION_THRESHOLD,
    PRIMARY_METRICS,
    SECONDARY_METRICS,
    TRADITIONAL_MODELS,
    LOGISTIC_REGRESSION_MAX_ITER,
    XGBOOST_OBJECTIVE,
    XGBOOST_EVAL_METRIC
)


# ------------------------------------------------------------
# Runtime Environment Diagnostics
# ------------------------------------------------------------
# Report software versions and protocol parameters so that the
# experimental configuration is visible and reproducible.
print("Traditional ML baseline — revised protocol")
print("=" * 60)

print(f"Python:            {sys.version.split()[0]}")
print(f"NumPy:             {np.__version__}")
print(f"Pandas:            {pd.__version__}")
print(f"scikit-learn:      {sklearn.__version__}")
print(f"XGBoost:           {xgb.__version__}")

# Report the protocol version and project location used by this run.
print(f"Protocol:          {PROTOCOL_VERSION}")
print(f"Project code dir:  {CODE_DIR}")

# Report the key experimental parameters imported from the protocol.
print(f"Datasets:          {EXPECTED_DATASETS}")
print(f"Training sizes:    {TRAINING_SIZES}")
print(f"Subset seeds:      {SUBSET_SEEDS}")
print(f"Test size:         {TEST_SIZE}")
print(f"Test seed:         {TEST_SEED}")
print(f"Model seed:        {MODEL_SEED}")

print("=" * 60)
print("✓ Project modules loaded successfully.")

Traditional ML baseline — revised protocol
Python:            3.12.13
NumPy:             2.0.2
Pandas:            2.3.3
scikit-learn:      1.6.1
XGBoost:           3.2.0
Protocol:          low_data_tabpfn_v1
Project code dir:  /kaggle/input/datasets/nicolausdeemese/iu-thesis-nicolas/files
Datasets:          ('diabetes', 'cardiovascular')
Training sizes:    (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
Subset seeds:      (42, 123, 456, 789, 1011)
Test size:         0.2
Test seed:         42
Model seed:        42
✓ Project modules loaded successfully.


## Load Prepared Experiment

In [2]:
# ============================================================
# Cell 2 — Locate and load the prepared experiment
# ============================================================

# Allow the prepared-data directory to be specified explicitly.
# This is useful when the notebook is executed in different environments.
PREPARED_DATA_DIR_ENV = os.environ.get("THESIS_PREPARED_DATA_DIR")

if PREPARED_DATA_DIR_ENV:

    # Resolve the explicitly provided path to an absolute path.
    PREPARED_DATA_DIR = Path(PREPARED_DATA_DIR_ENV).resolve()

else:

    candidates = []

    # ------------------------------------------------------------
    # Search local project structure
    # ------------------------------------------------------------
    # Check the standard local directory for the experiment manifest.
    local_processed = Path.cwd() / "data" / "processed"

    if (local_processed / "experiment_manifest.json").exists():
        candidates.append(local_processed)

    # ------------------------------------------------------------
    # Search Kaggle input structure
    # ------------------------------------------------------------
    # When running on Kaggle, search recursively for the manifest.
    if Path("/kaggle/input").exists():

        for manifest_file in Path("/kaggle/input").rglob(
            "experiment_manifest.json"
        ):
            # Store the directory containing the manifest as a candidate.
            candidates.append(manifest_file.parent)

    # Remove duplicate paths while preserving discovery order.
    candidates = list(
        dict.fromkeys(path.resolve() for path in candidates)
    )

    # Require exactly one valid prepared-data directory.
    if len(candidates) == 1:

        PREPARED_DATA_DIR = candidates[0]

    elif len(candidates) == 0:

        # Stop execution if the prepared experiment cannot be located.
        raise FileNotFoundError(
            "experiment_manifest.json was not found.\n\n"
            "Expected structure:\n"
            "/kaggle/input/<dataset>/prepared/"
            "experiment_manifest.json\n\n"
            "Set THESIS_PREPARED_DATA_DIR explicitly if necessary."
        )

    else:

        # Prevent accidental loading from an ambiguous experiment directory.
        raise RuntimeError(
            "Multiple experiment_manifest.json files were found.\n"
            "Set THESIS_PREPARED_DATA_DIR explicitly.\n\n"
            + "\n".join(str(path) for path in candidates)
        )


# ------------------------------------------------------------
# Load reproducible experiment artifact
# ------------------------------------------------------------

# Verify that the selected directory contains the expected manifest.
# The manifest acts as an integrity check before loading the experiment.
assert (
    PREPARED_DATA_DIR / "experiment_manifest.json"
).exists()

# Load the complete prepared experiment artifact.
# The loader reconstructs the data objects and metadata saved during
# the preprocessing/preparation stage.
experiment_data = load_prepared_experiment(
    PREPARED_DATA_DIR
)

# Extract the fixed training subsets used by the experimental protocol.
training_subsets = experiment_data["training_subsets"]

# Extract the fixed train/test split information.
split_data = experiment_data["split_data"]

# Extract the prepared datasets after preprocessing.
prepared_data = experiment_data["prepared_data"]

# Load the configuration used when the experiment artifact was created.
config = experiment_data["config"]

# Load the manifest containing experiment metadata and provenance.
manifest = experiment_data["manifest"]


# ------------------------------------------------------------
# Runtime confirmation
# ------------------------------------------------------------

print()
print("Prepared experiment loaded successfully.")
print(f"Prepared data dir: {PREPARED_DATA_DIR}")
print(f"Datasets:          {list(prepared_data.keys())}")


Prepared experiment loaded successfully.
Prepared data dir: /kaggle/input/datasets/nicolausdeemese/iu-thesis-nicolas/files/data/processed
Datasets:          ['diabetes', 'cardiovascular']


## Verify Experiment Protocol

In [3]:
# ============================================================
# Cell 3 — Verify manifest against the central protocol
# ============================================================

# Verify that the training sample sizes stored in the manifest
# match the centrally defined experimental protocol.
assert config["training_sizes"] == list(TRAINING_SIZES), (
    "Manifest training sizes do not match experiment_protocol.py."
)

# Verify that the subset seeds used to generate training subsets
# are identical to those defined in the central protocol.
assert config["training_seeds"] == list(SUBSET_SEEDS), (
    "Manifest subset seeds do not match experiment_protocol.py."
)

# Verify that the test-set size is consistent with the protocol.
assert config["test_size"] == TEST_SIZE, (
    "Manifest test size does not match experiment_protocol.py."
)

# Verify that the fixed test split uses the expected random seed.
assert config["test_seed"] == TEST_SEED, (
    "Manifest test seed does not match experiment_protocol.py."
)

# Confirm that the experiment is configured for binary classification.
assert config["n_classes"] == 2, (
    f"Expected binary classification (2 classes), "
    f"but manifest reports {config['n_classes']}."
)

# Verify that the prepared datasets are exactly the datasets
# specified by the central experimental protocol.
assert tuple(prepared_data.keys()) == EXPECTED_DATASETS, (
    "Prepared datasets do not match experiment_protocol.py."
)


# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

# Report the validated configuration so the experimental setup
# is visible before model training begins.
print("✓ Manifest and central protocol are consistent.")
print(f"✓ Datasets:        {EXPECTED_DATASETS}")
print(f"✓ Training sizes:  {TRAINING_SIZES}")
print(f"✓ Training seeds:  {SUBSET_SEEDS}")
print(f"✓ Test split:      {TEST_SIZE:.0%}, seed={TEST_SEED}")
print(f"✓ Classes:         {config['n_classes']}")

# The model seed controls model-level randomness and is intentionally
# separate from subset_seed, which controls training-subset selection.
print(f"✓ Model seed:      {MODEL_SEED} (independent of subset_seed)")

✓ Manifest and central protocol are consistent.
✓ Datasets:        ('diabetes', 'cardiovascular')
✓ Training sizes:  (8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)
✓ Training seeds:  (42, 123, 456, 789, 1011)
✓ Test split:      20%, seed=42
✓ Classes:         2
✓ Model seed:      42 (independent of subset_seed)


## Verify Training and Test Integrity

In [4]:
# ============================================================
# Cell 4 — Verify training/test integrity
# ============================================================

# Validate every dataset defined by the central experiment protocol.
for dataset_name in EXPECTED_DATASETS:

    # Retrieve the fixed test split for the current dataset.
    X_test = split_data[dataset_name]["X_test"]
    y_test = split_data[dataset_name]["y_test"]

    # ------------------------------------------------------------
    # Test-set integrity
    # ------------------------------------------------------------

    # Ensure that features and labels contain the same number of samples.
    assert len(X_test) == len(y_test)

    # Confirm that preprocessing produced no missing feature values.
    assert not X_test.isna().any().any()

    # Confirm that the target vector contains no missing labels.
    assert not y_test.isna().any()

    # Verify binary classification with labels encoded as 0 and 1.
    assert set(y_test.unique()) == {0, 1}

    # ------------------------------------------------------------
    # Training-subset integrity
    # ------------------------------------------------------------

    # Check every predefined training size and subset seed.
    # Each combination represents one fixed training subset.
    for training_size in TRAINING_SIZES:
        for subset_seed in SUBSET_SEEDS:

            subset = training_subsets[dataset_name][training_size][subset_seed]
            X_train = subset["X"]
            y_train = subset["y"]

            # Verify that the subset contains exactly the requested
            # number of training samples.
            assert len(X_train) == training_size
            assert len(y_train) == training_size

            # Training and test data must have the same feature space.
            assert X_train.shape[1] == X_test.shape[1]

            # Confirm that preprocessing produced no missing values
            # in either the training features or target.
            assert not X_train.isna().any().any()
            assert not y_train.isna().any()

            # Verify that both classes are represented in the subset.
            class_counts = y_train.value_counts()
            assert set(class_counts.index) == {0, 1}

            # Require equal numbers of samples from both classes.
            # A single unique class count indicates a balanced subset.
            assert class_counts.nunique() == 1

            # Ensure that no training sample is also present in the test set.
            # This prevents direct train/test overlap and data leakage.
            assert set(X_train.index).isdisjoint(set(X_test.index))

    # Summarize the integrity checks for the current dataset.
    print(
        f"✓ {dataset_name}: test={len(X_test):,}, "
        f"features={X_test.shape[1]}, balanced subsets verified"
    )

print("All integrity checks passed.")

✓ diabetes: test=45,895, features=21, balanced subsets verified
✓ cardiovascular: test=13,723, features=11, balanced subsets verified
All integrity checks passed.


## Define Evaluation Protocol

In [5]:
# ============================================================
# Cell 5 — Evaluation protocol
# ============================================================

from sklearn import metrics


def evaluate_predictions(
    y_true: pd.Series,
    y_proba: np.ndarray,
) -> dict[str, float]:

    # Convert inputs to NumPy arrays for consistent validation
    # and metric computation.
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba, dtype=float)

    # ------------------------------------------------------------
    # Input validation
    # ------------------------------------------------------------

    # Both inputs must be one-dimensional and have matching lengths.
    assert y_true.ndim == 1
    assert y_proba.ndim == 1
    assert len(y_true) == len(y_proba)

    # Confirm that the task is binary classification with labels 0 and 1.
    assert set(np.unique(y_true)) == {0, 1}

    # Predicted probabilities must be finite values in the [0, 1] range.
    assert np.isfinite(y_proba).all()
    assert ((y_proba >= 0.0) & (y_proba <= 1.0)).all()

    # Convert predicted probabilities into binary class predictions
    # using the threshold defined by the central experiment protocol.
    y_pred = (y_proba >= CLASSIFICATION_THRESHOLD).astype(int)

    # ------------------------------------------------------------
    # Metric computation
    # ------------------------------------------------------------

    return {
        # Threshold-independent ranking metrics.
        # These use predicted probabilities directly.
        "roc_auc": float(
            metrics.roc_auc_score(y_true, y_proba)
        ),
        "pr_auc": float(
            metrics.average_precision_score(y_true, y_proba)
        ),

        # Threshold-dependent classification metrics.
        # These use the binary predictions generated above.
        "accuracy": float(
            metrics.accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            metrics.balanced_accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            metrics.precision_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "recall": float(
            metrics.recall_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
        "f1": float(
            metrics.f1_score(
                y_true,
                y_pred,
                zero_division=0,
            )
        ),
    }


# Display the centrally defined evaluation configuration.
print(f"Primary metrics:   {PRIMARY_METRICS}")
print(f"Secondary metrics: {SECONDARY_METRICS}")
print(f"Threshold:         {CLASSIFICATION_THRESHOLD}")

Primary metrics:   ('roc_auc', 'pr_auc')
Secondary metrics: ('accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1')
Threshold:         0.5


## Define Traditional ML Models

In [6]:
# ============================================================
# Cell 6 — Traditional ML model factories
# ============================================================

# ------------------------------------------------------------
# Model factory functions
# ------------------------------------------------------------
# Each factory returns a fresh model instance for an experiment run.
# This prevents fitted model state from being reused across subsets
# or datasets.


def create_logistic_regression():
    # Standardize features before fitting Logistic Regression.
    # The scaler is part of the pipeline to ensure that scaling is
    # fitted only on the current training data.
    return Pipeline([
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                random_state=MODEL_SEED,
                max_iter=LOGISTIC_REGRESSION_MAX_ITER,
            ),
        ),
    ])


def create_random_forest():
    # Random Forest does not require feature scaling.
    return RandomForestClassifier(
        random_state=MODEL_SEED,
    )


def create_xgboost():
    # XGBoost is configured using the parameters defined in the
    # central experiment protocol.
    return xgb.XGBClassifier(
        objective=XGBOOST_OBJECTIVE,
        eval_metric=XGBOOST_EVAL_METRIC,
        random_state=MODEL_SEED,
    )


# Map protocol model names to their corresponding factory functions.
# Storing functions rather than fitted models ensures a fresh estimator
# is created whenever a new training run starts.
MODEL_FACTORIES = {
    "logistic_regression": create_logistic_regression,
    "random_forest": create_random_forest,
    "xgboost": create_xgboost,
}

# Preserve the configured model order for subsequent experiment loops.
MODEL_NAMES = tuple(MODEL_FACTORIES.keys())


# Verify that the notebook implements exactly the models specified
# by the central experimental protocol.
assert MODEL_NAMES == TRADITIONAL_MODELS, (
    f"Notebook models {MODEL_NAMES} do not match "
    f"experiment_protocol.py: {TRADITIONAL_MODELS}"
)


# Report the configured models and the model-level random seed.
print("Models:", MODEL_NAMES)
print(f"Model random state: {MODEL_SEED}")

# The subset_seed controls which predefined training samples are selected.
# MODEL_SEED controls stochastic behavior within the model itself.
print("subset_seed is used only to select the predefined training subset.")

Models: ('logistic_regression', 'random_forest', 'xgboost')
Model random state: 42
subset_seed is used only to select the predefined training subset.


## Define Single-Run Experiment

In [7]:
# ============================================================
# Cell 7 — Single-run function
# ============================================================

def run_single_experiment(
    dataset_name: str,
    model_name: str,
    training_size: int,
    subset_seed: int,
) -> dict[str, object]:

    # ------------------------------------------------------------
    # Validate experiment configuration
    # ------------------------------------------------------------
    # Select only predefined datasets, models, training sizes,
    # and subset seeds from the central experimental protocol.
    assert dataset_name in EXPECTED_DATASETS
    assert model_name in MODEL_NAMES
    assert training_size in TRAINING_SIZES
    assert subset_seed in SUBSET_SEEDS

    # Retrieve the fixed training subset corresponding to the
    # dataset, sample size, and subset seed.
    subset = training_subsets[dataset_name][training_size][subset_seed]

    X_train, y_train = subset["X"], subset["y"]
    X_test = split_data[dataset_name]["X_test"]
    y_test = split_data[dataset_name]["y_test"]

    # Create a fresh, unfitted model for this experiment run.
    model = MODEL_FACTORIES[model_name]()

    # ------------------------------------------------------------
    # Training
    # ------------------------------------------------------------
    # Measure only the model fitting time.
    start = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start

    # ------------------------------------------------------------
    # Prediction
    # ------------------------------------------------------------
    # Measure the time required to generate test-set probabilities.
    start = time.perf_counter()

    raw_proba = model.predict_proba(X_test)

    prediction_time = time.perf_counter() - start

    # ------------------------------------------------------------
    # Verify class ordering
    # ------------------------------------------------------------
    # Ensure that probability column 1 consistently represents
    # the positive class (label 1) across all model implementations.
    classes = np.asarray(model.classes_)

    assert np.array_equal(classes, np.array([0, 1])), (
        f"Unexpected class ordering for "
        f"{dataset_name}/{model_name}: {classes}"
    )

    # Extract the predicted probability of the positive class.
    y_proba = raw_proba[:, 1]

    # ------------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------------
    # Combine protocol metadata, runtime measurements, and all
    # evaluation metrics into one result record.
    return {
        "protocol_version": PROTOCOL_VERSION,
        "dataset": dataset_name,
        "model": model_name,
        "training_size": int(training_size),
        "subset_seed": int(subset_seed),
        "model_seed": int(MODEL_SEED),
        "classification_threshold": float(CLASSIFICATION_THRESHOLD),
        "training_time": float(training_time),
        "prediction_time": float(prediction_time),

        # evaluate_predictions computes all standardized metrics
        # using the fixed evaluation protocol.
        **evaluate_predictions(y_test, y_proba),
    }


print("✓ Single-run function defined.")

✓ Single-run function defined.


## Run Pilot Experiment

In [8]:
# ============================================================
# Cell 8 — Pilot experiment (not part of final results)
# ============================================================

# Use one dataset, one training size, and one predefined subset
# to perform a lightweight end-to-end validation of the pipeline.
PILOT_DATASET = EXPECTED_DATASETS[0]
PILOT_TRAINING_SIZE = 128
PILOT_SUBSET_SEED = SUBSET_SEEDS[0]

# Store pilot outputs separately from the final experiment results.
pilot_results = []

print("Pilot run — does not touch final result storage")
print("=" * 60)
print(
    f"Dataset={PILOT_DATASET}, n={PILOT_TRAINING_SIZE}, "
    f"subset_seed={PILOT_SUBSET_SEED}, model_seed={MODEL_SEED}"
)

# Run every configured traditional ML model on the same
# dataset, training subset, and test set.
for model_name in MODEL_NAMES:
    result = run_single_experiment(
        PILOT_DATASET,
        model_name,
        PILOT_TRAINING_SIZE,
        PILOT_SUBSET_SEED,
    )
    pilot_results.append(result)

    # Display the main ranking metrics as a quick sanity check.
    print(
        f"✓ {model_name:20s} "
        f"ROC-AUC={result['roc_auc']:.4f}  "
        f"PR-AUC={result['pr_auc']:.4f}"
    )


# ------------------------------------------------------------
# Pilot validation
# ------------------------------------------------------------

# Confirm that one result was produced for every configured model.
assert len(pilot_results) == len(MODEL_NAMES)

# Confirm that all models used the protocol-defined model seed.
assert all(r["model_seed"] == MODEL_SEED for r in pilot_results)

# Confirm that all models used the same predefined training subset.
assert all(r["subset_seed"] == PILOT_SUBSET_SEED for r in pilot_results)

print("✓ Pilot complete.")

Pilot run — does not touch final result storage
Dataset=diabetes, n=128, subset_seed=42, model_seed=42
✓ logistic_regression  ROC-AUC=0.7823  PR-AUC=0.3791
✓ random_forest        ROC-AUC=0.7625  PR-AUC=0.3152
✓ xgboost              ROC-AUC=0.7379  PR-AUC=0.2883
✓ Pilot complete.


## Define Checkpoint Helpers

In [9]:
# ============================================================
# Cell 9 — Checkpoint helpers
# ============================================================

# Use the configured results directory or default to Kaggle's
# writable working directory.
RESULTS_DIR = Path(
    os.environ.get("THESIS_RESULTS_DIR", "/kaggle/working")
)

# Create the directory if it does not already exist.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Store intermediate experiment results in a binary checkpoint
# and the completed results in a CSV file.
CHECKPOINT_PATH = RESULTS_DIR / "traditional_ml_default_checkpoint.pkl"
RESULTS_CSV_PATH = RESULTS_DIR / "traditional_ml_default_results.csv"


# Define the required fields for every experiment result.
# This schema is used to detect incomplete or malformed result rows.
REQUIRED_RESULT_KEYS = {
    "protocol_version",
    "dataset",
    "model",
    "training_size",
    "subset_seed",
    "model_seed",
    "classification_threshold",
    "training_time",
    "prediction_time",
    *PRIMARY_METRICS,
    *SECONDARY_METRICS,
}


def atomic_pickle_dump(obj, path: Path) -> None:
    """
    Save an object to a pickle file using an atomic replacement.

    The object is first written to a temporary file and then moved
    to the target path. This reduces the risk of leaving a partially
    written checkpoint if execution is interrupted during saving.
    """
    # Create a temporary file in the same directory as the target.
    tmp_path = path.with_suffix(path.suffix + ".tmp")

    with open(tmp_path, "wb") as handle:
        pickle.dump(
            obj,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    # Replace the existing checkpoint only after the temporary file
    # has been written successfully.
    os.replace(tmp_path, path)


def validate_result_rows(
    rows: list[dict[str, object]],
) -> set[tuple[object, ...]]:

    # Store unique experiment identifiers to detect duplicates.
    keys = set()

    for row in rows:

        # Check that every result contains the complete expected schema.
        missing = REQUIRED_RESULT_KEYS - set(row)

        assert not missing, (
            f"Result row missing keys: {missing}"
        )

        # Ensure the result was generated under the current protocol.
        assert row["protocol_version"] == PROTOCOL_VERSION

        # Ensure the model-level random seed matches the central protocol.
        assert row["model_seed"] == MODEL_SEED

        # Ensure the classification threshold is consistent across runs.
        assert (
            row["classification_threshold"]
            == CLASSIFICATION_THRESHOLD
        )

        # Define the unique identity of one experiment configuration.
        # Dataset, model, training size, and subset seed together
        # identify one expected result row.
        key = (
            row["dataset"],
            row["model"],
            row["training_size"],
            row["subset_seed"],
        )

        # Prevent the same experiment configuration from being stored twice.
        assert key not in keys, (
            f"Duplicate result key: {key}"
        )

        keys.add(key)

    return keys


def load_valid_checkpoint(
    path: Path,
) -> list[dict[str, object]]:

    # A missing checkpoint is valid for a new experiment run.
    if not path.exists():
        return []

    # Load previously completed experiment results.
    with open(path, "rb") as handle:
        rows = pickle.load(handle)

    # Verify the expected checkpoint structure before using its contents.
    assert isinstance(rows, list), (
        "Checkpoint must contain a list of result dictionaries."
    )

    # Validate schema, protocol settings, and duplicate experiment keys.
    validate_result_rows(rows)

    return rows


# Report the configured output locations.
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"CSV output: {RESULTS_CSV_PATH}")

Checkpoint: /kaggle/working/traditional_ml_default_checkpoint.pkl
CSV output: /kaggle/working/traditional_ml_default_results.csv


## Run Full Experiment

In [10]:
# ============================================================
# Cell 10 — Full experiment: checkpoint / resume
# ============================================================

# ------------------------------------------------------------
# Define the complete experiment grid
# ------------------------------------------------------------
# Each unique combination of dataset, model, training size,
# and subset seed represents one independent experiment run.
EXPECTED_KEYS = {
    (dataset_name, model_name, training_size, subset_seed)
    for dataset_name in EXPECTED_DATASETS
    for model_name in MODEL_NAMES
    for training_size in TRAINING_SIZES
    for subset_seed in SUBSET_SEEDS
}

# Count the total number of expected runs in the experiment grid.
EXPECTED_RUNS = len(EXPECTED_KEYS)


# ------------------------------------------------------------
# Load previous results
# ------------------------------------------------------------

# Load and validate any existing checkpoint.
# If no checkpoint exists, an empty result list is returned.
results = load_valid_checkpoint(CHECKPOINT_PATH)

# Extract the unique experiment keys already completed.
completed_runs = validate_result_rows(results)

# Every checkpointed run must belong to the current experiment grid.
assert completed_runs <= EXPECTED_KEYS


# ------------------------------------------------------------
# Experiment status
# ------------------------------------------------------------

print("Traditional ML baseline experiment")
print("=" * 60)
print(f"Expected runs: {EXPECTED_RUNS}")
print(f"Completed from checkpoint: {len(completed_runs)}")
print(f"Remaining: {EXPECTED_RUNS - len(completed_runs)}")
print("=" * 60)


# Start timing only the current execution of this cell.
experiment_start = time.perf_counter()


# ------------------------------------------------------------
# Execute experiment grid
# ------------------------------------------------------------

# Iterate through every predefined experimental configuration.
for dataset_name in EXPECTED_DATASETS:
    for training_size in TRAINING_SIZES:
        for subset_seed in SUBSET_SEEDS:
            for model_name in MODEL_NAMES:

                # Unique identifier for the current experiment configuration.
                run_key = (
                    dataset_name,
                    model_name,
                    training_size,
                    subset_seed,
                )

                # Skip runs already stored in the checkpoint.
                # This allows the experiment to resume after interruption.
                if run_key in completed_runs:
                    continue

                # Train, predict, and evaluate one model configuration.
                result = run_single_experiment(
                    dataset_name,
                    model_name,
                    training_size,
                    subset_seed,
                )

                # Add the completed run to the in-memory result collection.
                results.append(result)
                completed_runs.add(run_key)

                # Validate the complete result collection before saving.
                validate_result_rows(results)

                # Save the updated checkpoint immediately after each run.
                # This minimizes lost progress if execution is interrupted.
                atomic_pickle_dump(
                    results,
                    CHECKPOINT_PATH,
                )

                # Maintain a human-readable CSV copy of the current results.
                pd.DataFrame(results).to_csv(
                    RESULTS_CSV_PATH,
                    index=False,
                )

                # Report progress and the main evaluation metrics.
                print(
                    f"[{len(results):3d}/{EXPECTED_RUNS}] "
                    f"{dataset_name:16s} | "
                    f"{model_name:20s} | "
                    f"n={training_size:5d} | "
                    f"subset_seed={subset_seed:4d} | "
                    f"ROC-AUC={result['roc_auc']:.4f} | "
                    f"PR-AUC={result['pr_auc']:.4f}"
                )


# ------------------------------------------------------------
# Final experiment validation
# ------------------------------------------------------------

# Measure the elapsed time for this execution of the experiment cell.
elapsed = time.perf_counter() - experiment_start

# Confirm that every expected configuration has been completed.
assert len(completed_runs) == EXPECTED_RUNS
assert completed_runs == EXPECTED_KEYS


# ------------------------------------------------------------
# Completion summary
# ------------------------------------------------------------

print("=" * 60)
print("✓ Full traditional ML baseline experiment completed.")
print(f"✓ Runs: {len(results)}")
print(f"✓ Elapsed: {elapsed:.2f} s")

Traditional ML baseline experiment
Expected runs: 330
Completed from checkpoint: 0
Remaining: 330
[  1/330] diabetes         | logistic_regression  | n=    8 | subset_seed=  42 | ROC-AUC=0.7039 | PR-AUC=0.2957
[  2/330] diabetes         | random_forest        | n=    8 | subset_seed=  42 | ROC-AUC=0.7052 | PR-AUC=0.2860
[  3/330] diabetes         | xgboost              | n=    8 | subset_seed=  42 | ROC-AUC=0.6080 | PR-AUC=0.1913
[  4/330] diabetes         | logistic_regression  | n=    8 | subset_seed= 123 | ROC-AUC=0.7393 | PR-AUC=0.3206
[  5/330] diabetes         | random_forest        | n=    8 | subset_seed= 123 | ROC-AUC=0.7252 | PR-AUC=0.3276
[  6/330] diabetes         | xgboost              | n=    8 | subset_seed= 123 | ROC-AUC=0.5000 | PR-AUC=0.1529
[  7/330] diabetes         | logistic_regression  | n=    8 | subset_seed= 456 | ROC-AUC=0.6084 | PR-AUC=0.2122
[  8/330] diabetes         | random_forest        | n=    8 | subset_seed= 456 | ROC-AUC=0.6579 | PR-AUC=0.2422
[  9/3

## Validate and Export Final Results

In [11]:
# ============================================================
# Cell 11 — Validate and export final results
# ============================================================

# Convert all completed runs into a tabular result structure.
results_df = pd.DataFrame(results)

# Verify the expected number of runs and experiment dimensions.
assert len(results_df) == EXPECTED_RUNS
assert set(results_df["dataset"]) == set(EXPECTED_DATASETS)
assert set(results_df["model"]) == set(MODEL_NAMES)
assert set(results_df["training_size"]) == set(TRAINING_SIZES)
assert set(results_df["subset_seed"]) == set(SUBSET_SEEDS)

# Verify that all runs use the same protocol and model seed.
assert set(results_df["model_seed"]) == {MODEL_SEED}
assert set(results_df["protocol_version"]) == {PROTOCOL_VERSION}

# Ensure the primary metrics were successfully computed for every run.
assert results_df["roc_auc"].notna().all()
assert results_df["pr_auc"].notna().all()

# Each dataset-model combination must contain every
# training-size and subset-seed configuration exactly once.
expected_per_group = len(TRAINING_SIZES) * len(SUBSET_SEEDS)

counts = results_df.groupby(
    ["dataset", "model"]
).size()

assert (counts == expected_per_group).all()

# Reconstruct the unique run keys and compare them with
# the complete expected experiment grid.
actual_keys = {
    (
        row.dataset,
        row.model,
        int(row.training_size),
        int(row.subset_seed),
    )
    for row in results_df.itertuples(index=False)
}

assert actual_keys == EXPECTED_KEYS

# Sort results into a stable order for reproducible inspection and export.
results_df = results_df.sort_values(
    [
        "dataset",
        "training_size",
        "subset_seed",
        "model",
    ]
).reset_index(drop=True)

# Define final output paths.
FINAL_CSV_PATH = RESULTS_DIR / "traditional_ml_default_results.csv"
FINAL_PKL_PATH = RESULTS_DIR / "traditional_ml_default_results.pkl"

# Export tabular results for analysis and a structured copy for later reuse.
results_df.to_csv(
    FINAL_CSV_PATH,
    index=False,
)

atomic_pickle_dump(
    results_df.to_dict("records"),
    FINAL_PKL_PATH,
)

print("Final result checks passed.")
print(f"Rows: {len(results_df)}")
print(f"Columns: {len(results_df.columns)}")
print(f"CSV: {FINAL_CSV_PATH}")
print(f"PKL: {FINAL_PKL_PATH}")

display(results_df.head(12))

Final result checks passed.
Rows: 330
Columns: 16
CSV: /kaggle/working/traditional_ml_default_results.csv
PKL: /kaggle/working/traditional_ml_default_results.pkl


,protocol_version,dataset,model,training_size,subset_seed,model_seed,classification_threshold,training_time,prediction_time,roc_auc,pr_auc,accuracy,balanced_accuracy,precision,recall,f1
0,low_data_tabpfn_v1,cardiovascular,logistic_regression,8,42,42,0.5,0.005045,0.003434,0.307954,0.377772,0.359397,0.358793,0.335633,0.300884,0.317310
1,low_data_tabpfn_v1,cardiovascular,random_forest,8,42,42,0.5,0.154958,0.030199,0.329778,0.393203,0.390002,0.390768,0.399746,0.464212,0.429574
2,low_data_tabpfn_v1,cardiovascular,xgboost,8,42,42,0.5,0.018186,0.005855,0.398484,0.453455,0.397945,0.398484,0.402979,0.450221,0.425292
3,low_data_tabpfn_v1,cardiovascular,logistic_regression,8,123,42,0.5,0.004665,0.003056,0.698921,0.706050,0.666472,0.665395,0.704189,0.562003,0.625113
4,low_data_tabpfn_v1,cardiovascular,random_forest,8,123,42,0.5,0.162426,0.028612,0.694971,0.668531,0.625884,0.624436,0.667679,0.485567,0.562244
5,low_data_tabpfn_v1,cardiovascular,xgboost,8,123,42,0.5,0.017805,0.005605,0.571123,0.535728,0.571377,0.571123,0.569653,0.546834,0.558010
6,low_data_tabpfn_v1,cardiovascular,logistic_regression,8,456,42,0.5,0.004383,0.003020,0.651324,0.626359,0.609779,0.609435,0.612232,0.576436,0.593795
7,low_data_tabpfn_v1,cardiovascular,random_forest,8,456,42,0.5,0.157484,0.028606,0.724357,0.711641,0.676674,0.674879,0.763023,0.502651,0.606055
8,low_data_tabpfn_v1,cardiovascular,xgboost,8,456,42,0.5,0.017249,0.006311,0.599263,0.556078,0.600015,0.599263,0.611064,0.527099,0.565984
9,low_data_tabpfn_v1,cardiovascular,logistic_regression,8,789,42,0.5,0.004648,0.003201,0.752071,0.732483,0.686730,0.685511,0.738191,0.568483,0.642316
